In [1]:
import json
import random
import base64
import os
import shutil
import subprocess
import dill
import nltk
import itertools
import io
import math
import textwrap
import taxonomy
import time
import numpy as np
import evaluate

from PIL import Image
from pathlib import Path
from collections import Counter

/Users/aadi.joshi/miniconda3/envs/general/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open(f'responses_judge_gpt_54.dill', 'rb') as fp:
    responses = dill.load(fp)

# LLM-as-judge

In [3]:
skip_count = {}
for paper_id, error_id, res in responses:
    if 'model_responses' in res:
        for model_response in res['model_responses']:
            if 'judgement' not in model_response:
                model_id = model_response['model_id']
                skip_count[model_id] = skip_count.get(model_id, 0) + 1
                continue
            try:
                judgement = model_response['judgement']['final_score']
            except:
                judgement = json.loads(model_response['judgement'])
                model_response['judgement'] = judgement

In [4]:
skip_count

{'qwen_3_vl_8b_thinking': 1}

In [5]:
results = []
counts = {}
skip_count = {}
for paper_id, error_id, res in responses:
    if res['valid']:
        counts['all'] = counts.get('all', 0) + 1
        counts[error_id] = counts.get(error_id, 0) + 1

    if 'model_responses' in res:
        for model_response in res['model_responses']:
            if 'judgement' not in model_response:
                model_id = model_response['model_id']
                skip_count[model_id] = skip_count.get(model_id, 0) + 1
                continue

            model_id = model_response['model_id']
            #add task_id here if we add other tasks
            input_type = model_response['input_type']
            score = model_response['judgement']['final_score']

            results.append((model_id, input_type, error_id, paper_id, score))

In [6]:
skip_count

{'qwen_3_vl_8b_thinking': 1}

In [7]:
# valid responses
for k,v in counts.items():
    if k == 'all':
        print(f'total: {v}')
    else:
        print(f'{base64.b64decode(k).decode():50}: {v}')

total: 270
methodology_result_conflict                       : 56
semantic_misinterpretation                        : 57
suboptimal_methodology                            : 55
contradiction_with_established_knowledge          : 47
logically_invalid_interpretation                  : 55


In [8]:
scores = {}

for model, input_type, error_id, paper_id, score in results:
    meta_id = f'{model}___{input_type}___{error_id}'
    scores[meta_id] = scores.get(meta_id, []) + [score]

In [9]:
for meta_id, score_array in sorted(scores.items(), key=lambda x: (x[0].split('___')[0], x[0].split('___')[1], x[0].split('___')[2])):
    avg_score = np.mean(score_array)
    model, input_type, error_id = meta_id.split('___')
    error_category = base64.b64decode(error_id).decode()
    print(f'{model:15} {input_type:20} {error_category:50} {avg_score:3.2}')

gemini_25_flash images               contradiction_with_established_knowledge           1.7
gemini_25_flash images               logically_invalid_interpretation                   2.1
gemini_25_flash images               methodology_result_conflict                        1.6
gemini_25_flash images               semantic_misinterpretation                         1.8
gemini_25_flash images               suboptimal_methodology                             1.3
gemini_25_flash images_and_latex     contradiction_with_established_knowledge           2.2
gemini_25_flash images_and_latex     logically_invalid_interpretation                   4.0
gemini_25_flash images_and_latex     methodology_result_conflict                        2.8
gemini_25_flash images_and_latex     semantic_misinterpretation                         2.8
gemini_25_flash images_and_latex     suboptimal_methodology                             2.4
gemini_25_flash latex                contradiction_with_established_knowledge   

In [10]:
input_types = ['images', 'latex', 'images_and_latex']
input_types_to_name = {
    'images': 'Images',
    'latex': 'Latex',
    'images_and_latex': r'Images \& Latex',
}

error_categories = ['contradiction_with_established_knowledge', 'logically_invalid_interpretation', 'methodology_result_conflict', 'semantic_misinterpretation', 'suboptimal_methodology']
error_cat_to_acronym = {
    'contradiction_with_established_knowledge': 'ConEK',
    'logically_invalid_interpretation': 'LogII',
    'methodology_result_conflict': 'MetRC',
    'semantic_misinterpretation': 'SemM',
    'suboptimal_methodology': 'SubM'
}

error_category_names = [taxonomy.TAXONOMY_DATA[cat]['name'] for cat in error_categories]
error_category_acronyms = [error_cat_to_acronym[cat] for cat in error_categories]

model_ids = ['gpt_54', 'gpt_54_mini', 'gpt_54_nano', 'gemini_25_flash', 'gemini_25_flash_lite', 'gemini_31_flash_lite', 'qwen_3_vl_30b_a3b_thinking', 'qwen_3_vl_8b_thinking', 'qwen_3_vl_8b_instruct']
model_id_to_name = {
    'gpt_54': 'GPT 5.4', 
    'gpt_54_mini': 'GPT 5.4-mini (high)',
    'gpt_54_nano': 'GPT 5.4-nano (high)',
    'gpt_54_low': 'GPT 5.4', 
    'gpt_54_mini_low': 'GPT 5.4-mini (low)',
    'gpt_54_nano_low': 'GPT 5.4-nano (low)',
    'gemini_25_flash': 'Gemini 2.5 Flash (low)',
    'gemini_25_flash_lite': 'Gemini 2.5 Flash-lite (low)',
    'gemini_31_flash_lite': 'Gemini 3.1 Flash-lite (low)',
    'qwen_3_vl_30b_a3b_thinking': 'Qwen3-VL 30BA3B Thinking',
    'qwen_3_vl_8b_thinking': 'Qwen3-VL 8B Thinking',
    'qwen_3_vl_8b_instruct': 'Qwen3-VL 8B Instruct',
}

print(r'\begin{tabular}{c|ccccc}')
print(r'    \toprule')
#print(r'    Model & \multicolumn{5}{|c|}{images} & \multicolumn{5}{|c|}{latex} & \multicolumn{5}{|c|}{images_and_latex} \\')
print(r'    Model & ' + ' & '.join(error_category_acronyms) + r' \\')
print(r'    \midrule')

for model_id in model_ids:
    model_name = model_id_to_name[model_id]
    model_row = f'   {model_name} & ' + '& ' * (len(error_category_acronyms)-1) + r' \\'
    print(model_row)
    print(r'    \midrule')
    for input_type in input_types:
        input_name = input_types_to_name[input_type]
        outp = f'   {input_name} & '
        for error_cat in error_categories:
            error_cat_b64 = base64.urlsafe_b64encode(error_cat.encode()).decode()
            score_array = scores[f'{model_id}___{input_type}___{error_cat_b64}']
            score = np.mean(score_array)
            outp += f'{score:.2} & '
        print(outp[:-2] + r'\\' )

    print(r'    \midrule')
print(r'\bottomrule')
print(r'\end{tabular}')

\begin{tabular}{c|ccccc}
    \toprule
    Model & ConEK & LogII & MetRC & SemM & SubM \\
    \midrule
   GPT 5.4 & & & & &  \\
    \midrule
   Images & 3.3 & 4.7 & 3.8 & 3.6 & 3.4 \\
   Latex & 3.7 & 4.8 & 4.3 & 3.5 & 3.2 \\
   Images \& Latex & 3.6 & 4.8 & 4.2 & 3.5 & 3.4 \\
    \midrule
   GPT 5.4-mini (high) & & & & &  \\
    \midrule
   Images & 2.3 & 4.1 & 3.3 & 2.7 & 2.2 \\
   Latex & 2.9 & 4.5 & 3.7 & 2.6 & 2.1 \\
   Images \& Latex & 2.9 & 4.2 & 3.7 & 2.9 & 2.4 \\
    \midrule
   GPT 5.4-nano (high) & & & & &  \\
    \midrule
   Images & 2.0 & 3.0 & 2.4 & 2.0 & 1.7 \\
   Latex & 2.4 & 4.3 & 3.3 & 2.4 & 1.6 \\
   Images \& Latex & 2.2 & 4.2 & 3.1 & 2.6 & 1.6 \\
    \midrule
   Gemini 2.5 Flash (low) & & & & &  \\
    \midrule
   Images & 1.7 & 2.1 & 1.6 & 1.8 & 1.3 \\
   Latex & 2.2 & 3.9 & 2.6 & 2.9 & 2.5 \\
   Images \& Latex & 2.2 & 4.0 & 2.8 & 2.8 & 2.4 \\
    \midrule
   Gemini 2.5 Flash-lite (low) & & & & &  \\
    \midrule
   Images & 1.7 & 2.0 & 1.9 & 1.9 & 1.5 \\
   Lat

# Surface-Based Similarity metrics

These metrics are based on token overlap.

* Rouge-L: Based on the longest common subsequence. Used frequentyly in summarization.
* BLEU: Frequently used for translartion.
* BERTScore: Contextualized representations for each token, then computes similarity.

def evaluate_generated(responses, metric_name, score_key):
    metric = evaluate.load(metric_name)
    results = []
    for i, data in enumerate(responses):
        paper_id, err_id, res = data
        if i % 20 == 0:
            print(i)
            
        if 'model_responses' in res:
            ground_truth = None
            for gr in res['ground_truth']:
                if gr['task_id'] == 'generate_ground_truth_description':
                    ground_truth = gr['response_text']
                    break

            for model_response in res['model_responses']:
                model_id = model_response['model_id']
                #add task_id here if we add other tasks
                input_type = model_response['input_type']

                generated = [
                    model_response['response_text'],
                ]

                references = [[
                    ground_truth,
                ]]
                
                score = metric.compute(predictions=generated, references=references)[score_key]

                results.append((model_id, input_type, error_id, paper_id, score))

    return results

In [27]:
def evaluate_generated2(responses, metric_name, score_key, additional_kwargs):
    metric = evaluate.load(metric_name)
    generated_with_gold = {}
    for paper_id, error_id, res in responses:
        if 'model_responses' in res:
            ground_truth = None
            for gr in res['ground_truth']:
                if gr['task_id'] == 'generate_ground_truth_description':
                    ground_truth = gr['response_text']
                    break
   
            if ground_truth is None:
                raise Exception('No ground truth')
            
            for model_response in res['model_responses']:
                model_id = model_response['model_id']
                input_type = model_response['input_type']
                meta_id = f'{model_id}___{input_type}___{error_id}'

                if meta_id not in generated_with_gold:
                    generated_with_gold[meta_id] = {
                        'generated': [],
                        'ground_truth': [],
                    }
                
                generated = model_response['response_text']
                if generated is None or len(generated) == 0:
                    continue

                generated_with_gold[meta_id]['generated'].append(generated)
                generated_with_gold[meta_id]['ground_truth'].append([ground_truth])

    scores = {}

    for i, item in enumerate(generated_with_gold.items()):
        if i % 20 == 0:
            print(i)

        meta_id, generated_and_gold = item
        generated = generated_and_gold['generated']
        references = generated_and_gold['ground_truth']
        score = metric.compute(predictions=generated, references=references, **additional_kwargs)[score_key]
        scores[meta_id] = scores.get(meta_id, []) + [score]

    return scores

In [12]:
def print_table(scores):
    print(r'\begin{tabular}{c|ccccc}')
    print(r'    \toprule')
    #print(r'    Model & \multicolumn{5}{|c|}{images} & \multicolumn{5}{|c|}{latex} & \multicolumn{5}{|c|}{images_and_latex} \\')
    print(r'    Model & ' + ' & '.join(error_category_acronyms) + r' \\')
    print(r'    \midrule')

    for model_id in model_ids:
        model_name = model_id_to_name[model_id]
        model_row = f'   {model_name} & ' + '& ' * (len(error_category_acronyms)-1) + r' \\'
        print(model_row)
        print(r'    \midrule')
        for input_type in input_types:
            input_name = input_types_to_name[input_type]
            outp = f'   {input_name} & '
            for error_cat in error_categories:
                error_cat_b64 = base64.urlsafe_b64encode(error_cat.encode()).decode()
                score_array = scores[f'{model_id}___{input_type}___{error_cat_b64}']
                score = np.mean(score_array)
                outp += f'{score:.2} & '
            print(outp[:-2] + r'\\' )

        print(r'    \midrule')
    print(r'\bottomrule')
    print(r'\end{tabular}')

In [13]:
def print_merged_table(metric1_scores, metric1_name, metric2_scores, metric2_name):
    print(r'\begin{tabular}{c|ccccc|ccccc}')
    print(r'    \toprule')
    #print(r'    Model & \multicolumn{5}{|c|}{images} & \multicolumn{5}{|c|}{latex} & \multicolumn{5}{|c|}{images_and_latex} \\')
    print(r' & \multicolumn{5}{c}{'+f'{metric1_name}'+r'} & \multicolumn{5}{c}{'+f'{metric2_name}'+r'} \\')
    print(r'    Model & ' + ' & '.join(error_category_acronyms) + ' & ' + ' & '.join(error_category_acronyms) + r' \\')
    print(r'    \midrule')

    for model_id in model_ids:
        model_name = model_id_to_name[model_id]
        model_row = f'   {model_name} & ' + '& ' * (len(error_category_acronyms)-1) + r' \\'
        print(model_row)
        print(r'    \midrule')
        for input_type in input_types:
            input_name = input_types_to_name[input_type]
            outp = f'   {input_name} & '
            for metric_scores in [metric1_scores, metric2_scores]:
                scores = metric_scores
                for error_cat in error_categories:
                    error_cat_b64 = base64.urlsafe_b64encode(error_cat.encode()).decode()
                    score_array = scores[f'{model_id}___{input_type}___{error_cat_b64}']
                    score = np.mean(score_array)
                    outp += f'{score:.2f} & '
            print(outp[:-2] + r'\\' )

        print(r'    \midrule')
    print(r'\bottomrule')
    print(r'\end{tabular}')

## Rouge

In [28]:
rouge = evaluate_generated2(responses, "rouge", "rougeL", {"use_aggregator": False})

0
20
40
60
80
100
120
140
160


In [29]:
rouge

{'gpt_54_nano_low___images___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0': [[0.0867208672086721,
   0.09923664122137404,
   0.10882708585247884,
   0.0844205679201842,
   0.1059190031152648,
   0.06384558277654045,
   0.08860759493670885,
   0.15537190082644628,
   0.0899100899100899,
   0.10612991765782251,
   0.14285714285714285,
   0.07973962571196094,
   0.10362694300518135,
   0.09099018733273863,
   0.08326253186066271,
   0.06879606879606881,
   0.06686930091185411,
   0.13194444444444445,
   0.10644831115660186,
   0.09836065573770492,
   0.08990536277602522,
   0.09534127843986999,
   0.1,
   0.09395973154362416,
   0.07761194029850746,
   0.11235955056179775,
   0.07586206896551724,
   0.08386508659981769,
   0.11313131313131314,
   0.09710550887021475,
   0.18448637316561844,
   0.09368191721132897,
   0.07566024268379729,
   0.19937694704049844,
   0.0798004987531172,
   0.07636363636363637,
   0.09793103448275861,
   0.09706959706959706,
   0.1016949152542373,
   0.11958146487294

In [30]:
print_table(rouge)

\begin{tabular}{c|ccccc}
    \toprule
    Model & ConEK & LogII & MetRC & SemM & SubM \\
    \midrule
   GPT 5.4 & & & & &  \\
    \midrule
   Images & 0.11 & 0.15 & 0.12 & 0.12 & 0.12 \\
   Latex & 0.11 & 0.13 & 0.12 & 0.12 & 0.12 \\
   Images \& Latex & 0.11 & 0.13 & 0.11 & 0.12 & 0.12 \\
    \midrule
   GPT 5.4-mini (high) & & & & &  \\
    \midrule
   Images & 0.13 & 0.19 & 0.15 & 0.15 & 0.14 \\
   Latex & 0.14 & 0.2 & 0.16 & 0.15 & 0.14 \\
   Images \& Latex & 0.13 & 0.17 & 0.15 & 0.14 & 0.13 \\
    \midrule
   GPT 5.4-nano (high) & & & & &  \\
    \midrule
   Images & 0.12 & 0.17 & 0.13 & 0.13 & 0.13 \\
   Latex & 0.13 & 0.18 & 0.14 & 0.13 & 0.12 \\
   Images \& Latex & 0.12 & 0.19 & 0.14 & 0.13 & 0.12 \\
    \midrule
   Gemini 2.5 Flash (low) & & & & &  \\
    \midrule
   Images & 0.087 & 0.086 & 0.088 & 0.091 & 0.098 \\
   Latex & 0.11 & 0.13 & 0.1 & 0.12 & 0.12 \\
   Images \& Latex & 0.096 & 0.13 & 0.11 & 0.12 & 0.11 \\
    \midrule
   Gemini 2.5 Flash-lite (low) & & & & &  \

## BERTScore

In [31]:
bertscore = evaluate_generated2(responses, "bertscore", "recall", {"lang": "en"})

0


/Users/aadi.joshi/miniconda3/envs/general/lib/python3.13/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


20
40
60
80
100
120
140
160


In [33]:
print_merged_table(rouge, 'Rouge-L', bertscore, 'BERTScore (Recall)')

\begin{tabular}{c|ccccc|ccccc}
    \toprule
 & \multicolumn{5}{c}{Rouge-L} & \multicolumn{5}{c}{BERTScore (Recall)} \\
    Model & ConEK & LogII & MetRC & SemM & SubM & ConEK & LogII & MetRC & SemM & SubM \\
    \midrule
   GPT 5.4 & & & & &  \\
    \midrule
   Images & 0.11 & 0.15 & 0.12 & 0.12 & 0.12 & 0.83 & 0.86 & 0.84 & 0.82 & 0.82 \\
   Latex & 0.11 & 0.13 & 0.12 & 0.12 & 0.12 & 0.84 & 0.86 & 0.85 & 0.83 & 0.83 \\
   Images \& Latex & 0.11 & 0.13 & 0.11 & 0.12 & 0.12 & 0.83 & 0.86 & 0.84 & 0.83 & 0.82 \\
    \midrule
   GPT 5.4-mini (high) & & & & &  \\
    \midrule
   Images & 0.13 & 0.19 & 0.15 & 0.15 & 0.14 & 0.83 & 0.86 & 0.84 & 0.83 & 0.83 \\
   Latex & 0.14 & 0.20 & 0.16 & 0.15 & 0.14 & 0.83 & 0.87 & 0.85 & 0.84 & 0.83 \\
   Images \& Latex & 0.13 & 0.17 & 0.15 & 0.14 & 0.13 & 0.83 & 0.86 & 0.84 & 0.83 & 0.82 \\
    \midrule
   GPT 5.4-nano (high) & & & & &  \\
    \midrule
   Images & 0.12 & 0.17 & 0.13 & 0.13 & 0.13 & 0.83 & 0.85 & 0.83 & 0.82 & 0.82 \\
   Latex & 0.13 & 

# Human-Validated Subset

In [34]:
def get_choice_value(annotation):
    annotations = annotation['annotations']
    for ann in annotations:
        if ann['was_cancelled']:
            return None
        else:
            for res in ann['result']:
                if res['type'] == 'choices' and res['from_name'] == 'edit_correct':
                    choice = res['value']['choices'][0]
                    if choice == 'yes':
                        return 1
                    else:
                        return 0

In [35]:
def print_merged_table(metric1_scores, metric1_name, metric2_scores, metric2_name, precision=1):
    print(r'\begin{tabular}{c|ccccc|ccccc}')
    print(r'    \toprule')
    #print(r'    Model & \multicolumn{5}{|c|}{images} & \multicolumn{5}{|c|}{latex} & \multicolumn{5}{|c|}{images_and_latex} \\')
    print(r' & \multicolumn{5}{c}{'+f'{metric1_name}'+r'} & \multicolumn{5}{c}{'+f'{metric2_name}'+r'} \\')
    print(r'    Model & ' + ' & '.join(error_category_acronyms) + ' & ' + ' & '.join(error_category_acronyms) + r' \\')
    print(r'    \midrule')

    for model_id in model_ids:
        model_name = model_id_to_name[model_id]
        model_row = f'   {model_name} & ' + '& ' * (len(error_category_acronyms)-1) + r' \\'
        print(model_row)
        print(r'    \midrule')
        for input_type in input_types:
            input_name = input_types_to_name[input_type]
            outp = f'   {input_name} & '
            for metric_scores in [metric1_scores, metric2_scores]:
                scores = metric_scores
                for error_cat in error_categories:
                    error_cat_b64 = base64.urlsafe_b64encode(error_cat.encode()).decode()
                    score_array = scores[f'{model_id}___{input_type}___{error_cat_b64}']
                    score = np.mean(score_array)
                    if precision == 1:
                        outp += f'{score:.1f} & '
                    else:
                        outp += f'{score:.2f} & '
            print(outp[:-2] + r'\\' )

        print(r'    \midrule')
    print(r'\bottomrule')
    print(r'\end{tabular}')

In [36]:
HUMAN_ANNOTATIONS_FILE_PATH = Path('./aj_main.json')
with open(HUMAN_ANNOTATIONS_FILE_PATH, 'r') as fp:
    annotations = json.load(fp)

In [37]:
human_validation = {}
for annotation in annotations:
    paper_id = annotation['data']['id']
    error_id = annotation['data']['error_id']
    meta_id = f'{paper_id}___{error_id}'
    annotation_value = get_choice_value(annotation)
    error_name = base64.urlsafe_b64decode(error_id).decode()
    human_validation[meta_id] = annotation_value

In [39]:
with open(f'responses_judge_gpt_54.dill', 'rb') as fp:
    responses = dill.load(fp)

for paper_id, error_id, res in responses:
    if 'model_responses' in res:
        for model_response in res['model_responses']:
            if 'judgement' not in model_response:
                continue

            try:
                judgement = model_response['judgement']['final_score']
            except:
                judgement = json.loads(model_response['judgement'])
                model_response['judgement'] = judgement

results = []
for paper_id, error_id, res in responses:
    if 'model_responses' in res:
        for model_response in res['model_responses']:
            if 'judgement' not in model_response:
                continue
            
            model_id = model_response['model_id']
            #add task_id here if we add other tasks
            input_type = model_response['input_type']
            score = model_response['judgement']['final_score']
            human_valid = human_validation.get(f'{paper_id}___{error_id}', None)
            results.append((model_id, input_type, error_id, paper_id, score, human_valid))

In [40]:
scores = {}

for model, input_type, error_id, paper_id, score, human_valid in results:
    meta_id = f'{model}___{input_type}___{error_id}'
    scores[meta_id] = scores.get(meta_id, []) + [score]

In [41]:
human_valid_scores = {}

for model, input_type, error_id, paper_id, score, human_valid in results:
    meta_id = f'{model}___{input_type}___{error_id}'
    if human_valid == 1:
        human_valid_scores[meta_id] = human_valid_scores.get(meta_id, []) + [score]

In [42]:
assert len(scores) == len(human_valid_scores.keys()) # model_input_errorid should be same

In [43]:
print_merged_table(scores, 'Full Dataset', human_valid_scores, 'Human Validated Subset')

\begin{tabular}{c|ccccc|ccccc}
    \toprule
 & \multicolumn{5}{c}{Full Dataset} & \multicolumn{5}{c}{Human Validated Subset} \\
    Model & ConEK & LogII & MetRC & SemM & SubM & ConEK & LogII & MetRC & SemM & SubM \\
    \midrule
   GPT 5.4 & & & & &  \\
    \midrule
   Images & 3.3 & 4.7 & 3.8 & 3.6 & 3.4 & 3.6 & 4.5 & 3.7 & 3.6 & 3.8 \\
   Latex & 3.7 & 4.8 & 4.3 & 3.5 & 3.2 & 4.0 & 4.7 & 4.1 & 3.8 & 3.4 \\
   Images \& Latex & 3.6 & 4.8 & 4.2 & 3.5 & 3.4 & 3.8 & 4.7 & 3.9 & 3.6 & 3.5 \\
    \midrule
   GPT 5.4-mini (high) & & & & &  \\
    \midrule
   Images & 2.3 & 4.1 & 3.3 & 2.7 & 2.2 & 2.4 & 3.6 & 3.1 & 2.8 & 2.4 \\
   Latex & 2.9 & 4.5 & 3.7 & 2.6 & 2.1 & 2.8 & 4.4 & 3.6 & 2.6 & 2.0 \\
   Images \& Latex & 2.9 & 4.2 & 3.7 & 2.9 & 2.4 & 3.0 & 3.9 & 3.6 & 3.1 & 2.8 \\
    \midrule
   GPT 5.4-nano (high) & & & & &  \\
    \midrule
   Images & 2.0 & 3.0 & 2.4 & 2.0 & 1.7 & 2.1 & 3.0 & 2.3 & 2.1 & 1.4 \\
   Latex & 2.4 & 4.3 & 3.3 & 2.4 & 1.6 & 2.6 & 4.3 & 3.2 & 2.6 & 1.4 \\
   Imag

In [44]:
rouge_score_human_valid = {}

#paper ids should be in same order]
filter_mask = {}
for model, input_type, error_id, paper_id, score, human_valid in results:
    meta_id = f'{model}___{input_type}___{error_id}'
    filter_mask[meta_id] = filter_mask.get(meta_id, []) + [human_valid == 1]

In [49]:
for k, v in filter_mask.items():
    print(k)
    print(len(v))
    print(len(bertscore[k][0]))
    print(len(rouge[k][0]))
    assert len(v) == len(bertscore[k][0]) == len(rouge[k][0])

gpt_54_nano_low___images___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0
55
55
55
gpt_54_nano_low___images_and_latex___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0
55
55
55
gpt_54_nano_low___latex___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0
55
55
55
gemini_25_flash___images___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0
55
55
55
gemini_25_flash___images_and_latex___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0
55
55
55
gemini_25_flash___latex___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0
55
55
55
gpt_54_mini_low___images___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0
55
55
55
gpt_54_mini_low___images_and_latex___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0
55
55
55
gpt_54_mini_low___latex___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0
55
55
55
gemini_25_flash_lite___images___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0
55
55
55
gemini_25_flash_lite___images_and_latex___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0
55
55
55
gemini_25_flash_lite___latex___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0
55
55
55
qwen_3_vl_8b_instruct___images___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0
55


AssertionError: 

In [51]:
bertscore_human_validated = {}
rouge_human_validated = {}
for k, v in filter_mask.items():
    bertscore_human_validated[k] = [s for s, valid in zip(bertscore[k][0], v) if valid]
    rouge_human_validated[k] = [s for s, valid in zip(rouge[k][0], v) if valid]

In [53]:
print_merged_table(rouge_human_validated, 'Rouge-L', bertscore_human_validated, 'BERTScore (Recall)', precision=2)

\begin{tabular}{c|ccccc|ccccc}
    \toprule
 & \multicolumn{5}{c}{Rouge-L} & \multicolumn{5}{c}{BERTScore (Recall)} \\
    Model & ConEK & LogII & MetRC & SemM & SubM & ConEK & LogII & MetRC & SemM & SubM \\
    \midrule
   GPT 5.4 & & & & &  \\
    \midrule
   Images & 0.11 & 0.15 & 0.11 & 0.13 & 0.12 & 0.83 & 0.85 & 0.84 & 0.83 & 0.82 \\
   Latex & 0.11 & 0.14 & 0.12 & 0.12 & 0.12 & 0.84 & 0.86 & 0.85 & 0.83 & 0.83 \\
   Images \& Latex & 0.10 & 0.13 & 0.11 & 0.12 & 0.12 & 0.84 & 0.85 & 0.84 & 0.83 & 0.83 \\
    \midrule
   GPT 5.4-mini (high) & & & & &  \\
    \midrule
   Images & 0.12 & 0.19 & 0.14 & 0.15 & 0.13 & 0.83 & 0.85 & 0.83 & 0.83 & 0.82 \\
   Latex & 0.13 & 0.20 & 0.15 & 0.15 & 0.13 & 0.83 & 0.87 & 0.85 & 0.84 & 0.82 \\
   Images \& Latex & 0.13 & 0.17 & 0.14 & 0.14 & 0.13 & 0.83 & 0.86 & 0.84 & 0.83 & 0.82 \\
    \midrule
   GPT 5.4-nano (high) & & & & &  \\
    \midrule
   Images & 0.12 & 0.17 & 0.13 & 0.13 & 0.12 & 0.83 & 0.85 & 0.83 & 0.82 & 0.82 \\
   Latex & 0.12 & 

In [ ]:
print_table(bertscore)

\begin{tabular}{c|ccccc}
    \toprule
    Model & ConEK & LogII & MetRC & SemM & SubM \\
    \midrule
   GPT 5.4 & & & & &  \\
    \midrule
   Images & 0.83 & 0.86 & 0.84 & 0.81 & 0.82 \\
   Latex & 0.84 & 0.86 & 0.85 & 0.83 & 0.83 \\
   Images \& Latex & 0.83 & 0.86 & 0.84 & 0.83 & 0.82 \\
    \midrule
   GPT 5.4-mini & & & & &  \\
    \midrule
   Images & 0.83 & 0.86 & 0.84 & 0.83 & 0.83 \\
   Latex & 0.83 & 0.87 & 0.85 & 0.84 & 0.83 \\
   Images \& Latex & 0.83 & 0.86 & 0.84 & 0.83 & 0.82 \\
    \midrule
   GPT 5.4-nano & & & & &  \\
    \midrule
   Images & 0.83 & 0.85 & 0.83 & 0.82 & 0.82 \\
   Latex & 0.84 & 0.87 & 0.85 & 0.84 & 0.83 \\
   Images \& Latex & 0.84 & 0.87 & 0.84 & 0.84 & 0.82 \\
    \midrule
   Gemini 2.5 Flash & & & & &  \\
    \midrule
   Images & 0.81 & 0.82 & 0.82 & 0.81 & 0.81 \\
   Latex & 0.83 & 0.84 & 0.83 & 0.83 & 0.83 \\
   Images \& Latex & 0.83 & 0.84 & 0.83 & 0.83 & 0.82 \\
    \midrule
   Gemini 2.5 Flash-lite & & & & &  \\
    \midrule
   Images & 0.8